# Reading a Transformer in Ten Lines — the runnable notebook

Run the cells from the top and it trains. Nothing to install.
**A CPU runtime is enough.**

This is the file a book in progress reads from the first line to the last. A
little over a hundred lines, with training, validation and generation all in
it, and no module-level variables. It is a substantial PyTorch rewrite built on
nanoGPT, which is public online.

Repository · https://github.com/YoungjuneKwon/ten-line-transformer

Youngjune Kwon

## 1. The data — one file of Shakespeare (about a million characters)

In [ ]:
!wget -q -O input.txt https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
print(open('input.txt').read()[:200])

## 2. The whole file

This is the file. The ten lines below are one training step, and each chapter of
the book opens one of them.

```python
# -- done once
train_data, val_data, c2i, i2c, V = load_data("input.txt")   # text -> array of ids
model = Transformer(V, T, C, H, n_block)                 # wte, wpe, blocks, wue

# -- repeated every step from here
tokens, answers = get_batch(train_data, B, T)   # (B, T), (B, T)      (1)
optimizer.zero_grad(set_to_none=True)
x = wte[tokens] + wpe[:T]                       # (B, T, C)           (2)
for blk in blocks:                              # n_block times       (3)
    x = x + blk.attn(layer_norm(x))             # (B, T, C)
    x = x + blk.mlp(layer_norm(x))              # (B, T, C)
logits = layer_norm(x) @ wue                    # (B, T, V)           (4)
loss = cross_entropy(logits, answers)           # one scalar
loss.backward()
optimizer.step()
```

Those ten lines are one step laid out flat. In the actual file they sit in two
places: the middle six are inside `Transformer.forward()` and the other four are
in the training loop in `train()`. Given the answers, `forward` returns the loss
along with the logits — the same arrangement as Hugging Face's
`GPT2LMHeadModel.forward(..., labels=None)` or nanoGPT's
`forward(idx, targets=None)`. Pass no answers and `None` comes back in the loss
position.

```python
  def forward(self, tokens, answers=None):
    x = self.wte[tokens] + self.wpe[:tokens.shape[1]]
    for blk in self.blocks:
      x = x + blk.attn(layer_norm(x))
      x = x + blk.mlp(layer_norm(x))
    logits = layer_norm(x) @ self.wue
    return logits, None if answers is None else cross_entropy(logits, answers)
```

```python
  for step in range(steps + 1):
    tokens, answers = get_batch(train_data, B, T)
    optimizer.zero_grad(set_to_none=True)
    _, loss = model(tokens, answers)
    loss.backward()
    optimizer.step()
```

They are the lines in the cell below. Not transcribed — taken from the same
file.

In [ ]:
import sys, math, time, torch, torch.nn as nn
from torch.nn import functional as F

def load_data(path, train_ratio=0.9):
  text = open(path, encoding="utf-8").read()
  chars = sorted(set(text))
  c2i = {c: i for i, c in enumerate(chars)}
  data = torch.tensor([c2i[c] for c in text], dtype=torch.long)
  cut = int(train_ratio * len(data))
  return data[:cut], data[cut:], c2i, {i: c for c, i in c2i.items()}, len(chars)

def get_batch(data, B, T):
  starts = torch.randint(len(data) - T - 1, (B,))
  tokens = torch.stack([data[s:s + T] for s in starts])
  answers = torch.stack([data[s + 1:s + T + 1] for s in starts])
  return tokens, answers

def layer_norm(x, eps=1e-5):
  return (x - x.mean(-1, keepdim=True)) / torch.sqrt(x.var(-1, unbiased=False, keepdim=True) + eps)

def cross_entropy(logits, answers):
  return F.cross_entropy(logits.reshape(-1, logits.shape[-1]), answers.reshape(-1))

class Attention(nn.Module):
  def __init__(self, T, C, H):
    super().__init__()
    self.H = H
    self.hs = C // H
    self.qkv = nn.Linear(C, 3 * H * self.hs, bias=False)
    self.proj = nn.Linear(C, C, bias=False)
    self.register_buffer("mask", torch.tril(torch.ones(T, T)).view(1, 1, T, T))

  def forward(self, x):
    b, t, c = x.shape
    q, k, v = [z.view(b, t, self.H, self.hs).transpose(1, 2)
               for z in self.qkv(x).split(c, dim=2)]
    att = (q @ k.transpose(-2, -1)) / math.sqrt(self.hs)
    att = att.masked_fill(self.mask[:, :, :t, :t] == 0, float("-inf"))
    y = F.softmax(att, dim=-1) @ v
    y = y.transpose(1, 2).contiguous().view(b, t, c)
    return self.proj(y)

class MLP(nn.Module):
  def __init__(self, C):
    super().__init__()
    self.fc = nn.Linear(C, 4 * C, bias=False)
    self.out = nn.Linear(4 * C, C, bias=False)

  def forward(self, x):
    return self.out(F.gelu(self.fc(x)))

class Block(nn.Module):
  def __init__(self, T, C, H):
    super().__init__()
    self.attn = Attention(T, C, H)
    self.mlp = MLP(C)

class Transformer(nn.Module):
  def __init__(self, V, T, C, H, n_block):
    super().__init__()
    self.wte = nn.Parameter(torch.randn(V, C) * 0.02)
    self.wpe = nn.Parameter(torch.randn(T, C) * 0.02)
    self.wue = nn.Parameter(torch.randn(C, V) * 0.02)
    self.blocks = nn.ModuleList([Block(T, C, H) for _ in range(n_block)])

  def forward(self, tokens, answers=None):
    x = self.wte[tokens] + self.wpe[:tokens.shape[1]]
    for blk in self.blocks:
      x = x + blk.attn(layer_norm(x))
      x = x + blk.mlp(layer_norm(x))
    logits = layer_norm(x) @ self.wue
    return logits, None if answers is None else cross_entropy(logits, answers)

def report(model, step, loss, t0, val_batch):
  val_tokens, val_answers = val_batch
  with torch.no_grad():
    _, val_loss = model(val_tokens, val_answers)
  print(f"{step:6d}{loss.item():10.4f}{val_loss.item():10.4f}{time.time() - t0:8.1f}")

def train(model, train_data, val_data, steps, B, T):
  optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4,
                                betas=(0.9, 0.95), weight_decay=0.1)
  print(f"{'step':>6}{'train':>10}{'val':>10}{'sec':>8}")
  report_every = max(1, steps // 10)
  t0 = time.time()
  for step in range(steps + 1):
    tokens, answers = get_batch(train_data, B, T)
    optimizer.zero_grad(set_to_none=True)
    _, loss = model(tokens, answers)
    loss.backward()
    optimizer.step()
    if step % report_every == 0:
      val_batch = get_batch(val_data, B, T)
      report(model, step, loss, t0, val_batch)

def generate(model, n_chars, T, c2i, i2c, seed_text="\n", temperature=0.8):
  seq = torch.tensor([[c2i[c] for c in seed_text]], dtype=torch.long)
  with torch.no_grad():
    for _ in range(n_chars):
      logits, _ = model(seq[:, -T:])
      probs = F.softmax(logits[:, -1, :] / temperature, dim=-1)
      seq = torch.cat([seq, torch.multinomial(probs, 1)], dim=1)
  return "".join(i2c[i] for i in seq[0].tolist())

def cli():
  steps = int(sys.argv[1]) if len(sys.argv) > 1 else 2000
  corpus = sys.argv[2] if len(sys.argv) > 2 else "input.txt"
  return steps, corpus

def main(steps, corpus):
  B, T, C, H, n_block = 32, 64, 128, 4, 4
  assert C % H == 0, "C must be divisible by H"
  torch.manual_seed(1337)

  train_data, val_data, c2i, i2c, V = load_data(corpus)
  model = Transformer(V, T, C, H, n_block)
  train(model, train_data, val_data, steps, B, T)
  print("\nsample")
  print(generate(model, 800, T, c2i, i2c))
  return model

## 3. Run it — 200 steps

It starts at `0: 4.2006 / 3.9836` and comes down to `200: 2.5400 / 2.4785`.
The seed is fixed at 1337, so you get the numbers printed in the README.

The `sec` column differs by machine. The loss values do not.

In [ ]:
model = main(200, 'input.txt')

## 4. 2000 steps

Shakespeare starts to show through. A model trained only to guess the next
single character puts a colon after a name and breaks the line.

One thing to expect: step 200 reads `2.4812` here, not `2.5400`. Reports land
every 200 steps instead of every 20, and each report draws a validation batch
from the same generator the training batches come from — so the draws differ.
Same seed, same code, different schedule. Compare runs of the same command.

In [ ]:
model = main(2000, 'input.txt')

## 5. Change something

The five numbers on the first line of `main` are the entire size of this model.
Drop `C` to 64 and the parameter count falls to a quarter; raise it to 256 and
the loss goes lower but it takes about five times as long.

In [ ]:
# B, T, C, H, n_block = 32, 64, 128, 4, 4   <- the defaults inside main
# Edit the line below and see for yourself.
import torch, time
B, T, C, H, n_block = 32, 64, 64, 4, 4
torch.manual_seed(1337)
train_data, val_data, c2i, i2c, V = load_data('input.txt')
m = Transformer(V, T, C, H, n_block)
print('parameters', sum(p.numel() for p in m.parameters()))
train(m, train_data, val_data, 200, B, T)